# Predict with SomaticXGB

Run from `4.SomaticXGB-predict/`.

1. **Build the feature matrix** with the same script as training: `../3.SomaticXGB-training/build_feature_matrix.R` (reads each pair’s step-2 `config.json`).
2. **Load a trained model** and score every row.
3. **Write** `predictions.tsv.gz` here (and an optional quick PR check if `label` is present).

No separate predict config and no parallel feature-extraction path.



## 0. Paths and packages

Edit `pair_dirs` (test / held-out pair working directories) and `model_path`.



In [ ]:
library(data.table)
library(xgboost)
library(PRROC)

# Working directory should be 4.SomaticXGB-predict/
stopifnot(file.exists("predict_SomaticXGB.ipynb"))

train_dir <- normalizePath("../3.SomaticXGB-training")
build_script <- file.path(train_dir, "build_feature_matrix.R")
stopifnot(file.exists(build_script))

# Pipeline-2 pair dirs to score (each with config.json + force-calling outputs)
pair_dirs <- c(
  "/path/to/simulations/test/45",
  "/path/to/simulations/test/54"
)

feature_matrix <- "feature_matrix.tsv.gz"
predictions_out <- "predictions.tsv.gz"
# Released model; or e.g. ../3.SomaticXGB-training/SomaticXGB_retrained
model_path <- "../models/SomaticXGB"

getwd()



## 1. Build the feature matrix (reuse step 3)

Calls the **same** `build_feature_matrix.R` used for training. Output is written under this directory.



In [ ]:
missing_dirs <- pair_dirs[!dir.exists(pair_dirs)]
if (length(missing_dirs)) {
  stop("pair_dirs not found:\n  ", paste(missing_dirs, collapse = "\n  "))
}

cmd <- paste(
  "Rscript",
  shQuote(build_script),
  paste(shQuote(pair_dirs), collapse = " "),
  shQuote(feature_matrix)
)
message("Running: ", cmd)
status <- system(cmd)
if (status != 0) stop("build_feature_matrix.R failed with status ", status)

dtm <- fread(feature_matrix)
message("Rows: ", nrow(dtm),
        if ("label" %in% names(dtm)) paste0("  positives: ", sum(dtm$label == 1, na.rm = TRUE)) else "")
if (!is.null(dtm$group)) print(table(dtm$group, useNA = "ifany"))
head(dtm)



## 2. Load model and predict

Uses the feature columns defined in step 3 (`SOMATICXGB_FEATURES` / training notebook).



In [ ]:
source(file.path(train_dir, "SomaticXGB_features.R"))

if (!file.exists(model_path)) {
  stop("Model not found: ", model_path)
}
model <- xgb.load(model_path)

miss <- setdiff(SOMATICXGB_FEATURES, names(dtm))
if (length(miss)) stop("Feature matrix missing columns: ", paste(miss, collapse = ", "))

dtm$pred <- predict(model, data.matrix(dtm[, SOMATICXGB_FEATURES, with = FALSE]))
fwrite(dtm, predictions_out, sep = "\t")
message("Wrote ", predictions_out)
summary(dtm$pred)



## 3. Optional: in-matrix PR curve (only if labels exist)

If `posfreq` was set in the step-2 configs, `label` is present and this is a quick check — not a substitute for a full benchmark report.



In [ ]:
if ("label" %in% names(dtm) && !all(is.na(dtm$label))) {
  pr <- pr.curve(scores.class0 = dtm$pred, weights.class0 = dtm$label, curve = TRUE)
  print(pr$auc.integral)
  plot(pr, main = "Precision-Recall (feature matrix labels)",
       xlab = "Recall", ylab = "Precision")
} else {
  message("No label column (or all NA); skip PR curve.")
}

